# Week 13 — Factor Models and Portfolio Optimization

**Course:** IEOR E4733 — Algorithmic Trading  
**Reference:** *Machine Learning for Algorithmic Trading*, Stefan Jansen — Chapters 4 (pp. 81–131), 5 (pp. 132–172), 13 (pp. 407–425)

## Overview

This notebook demonstrates the complete pipeline from signal evaluation to portfolio construction
using synthetic data. All data is generated synthetically — no external downloads required.

| Section | Topic | Key Concept |
|---------|-------|-------------|
| 1 | Imports & Setup | Libraries and configuration |
| 2 | Synthetic Data | Generate 100 stocks with 5 factor exposures |
| 3 | Signal Evaluation | IC, ICIR, quantile analysis for each factor |
| 4 | Signal Combination | IC-weighted composite signal |
| 5 | PCA & Eigenportfolios | Statistical factor model, scree plot |
| 6 | Mean-Variance Optimization | Efficient frontier, max Sharpe, min variance |
| 7 | Strategy Comparison | Equal weight vs max Sharpe vs min variance vs risk parity |
| 8 | Key Takeaways | Summary of findings |

## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.optimize import minimize
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({"figure.figsize": (14, 6), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

np.random.seed(42)
print("Setup complete.")

## 2. Generate Synthetic Stock Data

We create a synthetic universe of 100 stocks over 750 trading days (~3 years).
Each stock has exposures to 5 fundamental factors (value, momentum, quality, size, volatility)
that partially predict forward returns. This mimics a realistic equity factor model.

**Data-generating process:**
- A common market factor drives correlated returns
- 5 stock-level characteristics (factors) have weak but nonzero predictive power
- Idiosyncratic noise dominates (as in real markets)

In [ ]:
# --- Parameters ---
N_STOCKS = 100
N_DAYS = 750
FORWARD_DAYS = 5

# --- Common market factor ---
market_factor = np.random.normal(0.0003, 0.012, N_DAYS)

# --- Stock-level characteristics (static for simplicity, refreshed quarterly) ---
# These represent: value, momentum, quality, size, volatility scores
np.random.seed(42)
factor_names = ['Value', 'Momentum', 'Quality', 'Size', 'LowVol']

# Generate stable factor exposures for each stock
# (In practice these would be time-varying; here they change quarterly)
quarter_length = N_DAYS // 4

stock_names = [f'S{i:03d}' for i in range(N_STOCKS)]

# Factor exposures: each stock gets a score for each factor
all_exposures = {}
for q in range(4):
    exposures_q = np.random.randn(N_STOCKS, 5)
    start = q * quarter_length
    end = (q + 1) * quarter_length if q < 3 else N_DAYS
    for t in range(start, end):
        all_exposures[t] = exposures_q

# Factor return premiums (how much each factor predicts returns)
# These are small but nonzero, matching real-world signal strength
factor_premiums = np.array([0.0003, 0.0005, 0.0002, -0.0002, 0.0003])

# --- Generate daily returns ---
returns_matrix = np.zeros((N_DAYS, N_STOCKS))
betas = np.random.uniform(0.6, 1.4, N_STOCKS)
idio_vols = np.random.uniform(0.008, 0.022, N_STOCKS)

for t in range(N_DAYS):
    exposures_t = all_exposures[t]
    # Return = market beta * market + factor premium * exposures + noise
    factor_return = exposures_t @ factor_premiums
    idio_noise = np.random.randn(N_STOCKS) * idio_vols
    returns_matrix[t] = betas * market_factor[t] + factor_return + idio_noise

dates = pd.bdate_range('2022-01-03', periods=N_DAYS, freq='B')
returns_df = pd.DataFrame(returns_matrix, index=dates, columns=stock_names)
returns_df.index.name = 'date'

print(f"Returns panel: {returns_df.shape[0]} days x {returns_df.shape[1]} stocks")
print(f"Date range: {returns_df.index[0].date()} to {returns_df.index[-1].date()}")
returns_df.describe().T.head(10)

In [ ]:
# --- Build factor signals and forward returns ---
# For each stock and date, we have 5 factor signals and a forward 5-day return

records = []
for t in range(N_DAYS - FORWARD_DAYS):
    exposures_t = all_exposures[t]
    fwd_ret = returns_df.iloc[t+1:t+1+FORWARD_DAYS].sum()  # forward 5-day return
    for i in range(N_STOCKS):
        record = {
            'date': dates[t],
            'stock': stock_names[i],
            'forward_return': fwd_ret.iloc[i],
        }
        for k, fname in enumerate(factor_names):
            record[fname] = exposures_t[i, k]
        records.append(record)

panel = pd.DataFrame(records)
panel = panel.set_index(['date', 'stock']).sort_index()

print(f"Panel shape: {panel.shape}")
print(f"Columns: {list(panel.columns)}")
panel.head(10)

## 3. Signal Evaluation: IC and Quantile Analysis

For each factor, we compute:
- **IC (Information Coefficient):** Spearman rank correlation between the factor signal and
  forward returns, computed cross-sectionally at each date.
- **ICIR (IC Information Ratio):** mean(IC) / std(IC) — measures consistency.
- **Quantile returns:** Sort stocks into quintiles by signal, compute mean forward return per quintile.

In [ ]:
# --- Compute IC for each factor at each date ---

ic_results = {fname: [] for fname in factor_names}
ic_dates = []

unique_dates = panel.index.get_level_values('date').unique()

for date in unique_dates:
    cross_section = panel.loc[date]
    fwd = cross_section['forward_return']
    for fname in factor_names:
        signal = cross_section[fname]
        ic_val, _ = spearmanr(signal, fwd)
        ic_results[fname].append(ic_val)
    ic_dates.append(date)

ic_df = pd.DataFrame(ic_results, index=ic_dates)
ic_df.index.name = 'date'

# --- Summary statistics ---
ic_summary = pd.DataFrame({
    'Mean IC': ic_df.mean(),
    'Std IC': ic_df.std(),
    'ICIR': ic_df.mean() / ic_df.std(),
    't-stat': ic_df.mean() / ic_df.std() * np.sqrt(len(ic_df)),
    'IC > 0 (%)': (ic_df > 0).mean() * 100,
})

print("=" * 60)
print("SIGNAL EVALUATION SUMMARY")
print("=" * 60)
print(ic_summary.round(4).to_string())
print(f"\nNumber of cross-sectional dates: {len(ic_df)}")

In [ ]:
# --- Plot IC time series ---

fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for idx, fname in enumerate(factor_names):
    ax = axes[idx]
    ic_series = ic_df[fname]
    ax.bar(ic_series.index, ic_series.values, alpha=0.4, color=colors[idx], width=2)
    ax.axhline(0, color='black', lw=0.8)
    ax.axhline(ic_series.mean(), color='red', lw=2, ls='--',
               label=f'Mean IC = {ic_series.mean():.4f}')
    ax.set_title(f'{fname} — ICIR = {ic_series.mean()/ic_series.std():.3f}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('IC')
    ax.legend(fontsize=9)
    ax.set_ylim(-0.35, 0.35)

# Remove unused subplot
axes[5].set_visible(False)

fig.suptitle('Information Coefficient (IC) Time Series by Factor',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Quantile Analysis ---
# For each factor, sort stocks into quintiles and compute mean forward return per quintile

n_quantiles = 5

quantile_returns = {fname: {q: [] for q in range(1, n_quantiles + 1)}
                    for fname in factor_names}

for date in unique_dates:
    cs = panel.loc[date]
    for fname in factor_names:
        # Assign quintiles
        cs_sorted = cs.sort_values(fname)
        n = len(cs_sorted)
        quintile_size = n // n_quantiles
        for q in range(1, n_quantiles + 1):
            start_idx = (q - 1) * quintile_size
            end_idx = q * quintile_size if q < n_quantiles else n
            mean_ret = cs_sorted['forward_return'].iloc[start_idx:end_idx].mean()
            quantile_returns[fname][q].append(mean_ret)

# Compute average across all dates
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=True)

for idx, fname in enumerate(factor_names):
    ax = axes[idx]
    q_means = [np.mean(quantile_returns[fname][q]) * 252 * 100
               for q in range(1, n_quantiles + 1)]  # annualized, in percent
    q_labels = [f'Q{q}' for q in range(1, n_quantiles + 1)]
    bar_colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, n_quantiles))
    ax.bar(q_labels, q_means, color=bar_colors, edgecolor='white', lw=0.5)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(fname, fontsize=11, fontweight='bold')
    ax.set_xlabel('Quintile')
    spread = q_means[-1] - q_means[0]
    ax.annotate(f'Spread: {spread:.1f}%', xy=(0.5, 0.02),
                xycoords='axes fraction', ha='center', fontsize=8,
                color='darkblue', fontweight='bold')

axes[0].set_ylabel('Ann. Forward Return (%)')
fig.suptitle('Quantile Analysis: Mean Forward Return by Signal Quintile',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 4. IC-Weighted Signal Combination

We combine the 5 individual factor signals into a single composite alpha score.
The weight of each factor is proportional to its historical mean IC.
Factors with negative or very small IC receive less weight.

In [ ]:
# --- IC-weighted signal combination ---

# Use mean IC as weights (clip negatives to zero)
ic_weights_raw = ic_df.mean().clip(lower=0)
ic_weights = ic_weights_raw / ic_weights_raw.sum()

print("IC Weights for Signal Combination:")
print(ic_weights.round(4).to_string())
print()

# Compute composite signal for each date/stock
panel['Composite'] = sum(ic_weights[fname] * panel[fname]
                         for fname in factor_names)

# Compute IC for composite signal
composite_ic = []
for date in unique_dates:
    cs = panel.loc[date]
    ic_val, _ = spearmanr(cs['Composite'], cs['forward_return'])
    composite_ic.append(ic_val)

composite_ic = pd.Series(composite_ic, index=ic_dates)

print(f"Composite Signal Performance:")
print(f"  Mean IC:  {composite_ic.mean():.4f}")
print(f"  Std IC:   {composite_ic.std():.4f}")
print(f"  ICIR:     {composite_ic.mean() / composite_ic.std():.4f}")
print(f"  t-stat:   {composite_ic.mean() / composite_ic.std() * np.sqrt(len(composite_ic)):.2f}")
print(f"  IC > 0:   {(composite_ic > 0).mean() * 100:.1f}%")

In [ ]:
# --- Compare individual vs composite IC ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: ICIR comparison
ax = axes[0]
all_names = factor_names + ['Composite']
all_icir = [ic_df[f].mean() / ic_df[f].std() for f in factor_names]
all_icir.append(composite_ic.mean() / composite_ic.std())
bar_colors = colors + ['black']
bars = ax.barh(all_names, all_icir, color=bar_colors, alpha=0.8)
ax.axvline(0.5, color='red', ls='--', lw=1.5, label='ICIR = 0.5 threshold')
ax.set_xlabel('ICIR', fontsize=12)
ax.set_title('ICIR: Individual Factors vs Composite', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

# Right: Mean IC comparison
ax = axes[1]
all_mean_ic = list(ic_df.mean().values) + [composite_ic.mean()]
ax.barh(all_names, all_mean_ic, color=bar_colors, alpha=0.8)
ax.axvline(0.02, color='red', ls='--', lw=1.5, label='IC = 0.02 threshold')
ax.set_xlabel('Mean IC', fontsize=12)
ax.set_title('Mean IC: Individual Factors vs Composite', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 5. PCA and Eigenportfolios

We apply Principal Component Analysis to the stock return covariance matrix.
PCA extracts the dominant sources of co-movement (statistical factors)
without requiring any prior economic assumptions.

- **PC1** typically represents the market factor (roughly equal positive weights)
- **Eigenportfolios** are portfolios whose weights equal the eigenvector components
- The **scree plot** shows how many components are needed to explain most of the variance

In [ ]:
# --- PCA on return covariance matrix ---

# Demean returns
returns_demeaned = returns_df - returns_df.mean()

# Fit PCA
n_components = 20
pca = PCA(n_components=n_components)
pca.fit(returns_demeaned)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# How many components for 80%?
n_for_80 = np.argmax(cumulative_var >= 0.80) + 1

print(f"Explained variance by first {n_components} components:")
for i in range(min(10, n_components)):
    print(f"  PC{i+1}: {explained_var[i]:.4f}  (cumulative: {cumulative_var[i]:.4f})")
print(f"\nComponents needed for 80% variance: {n_for_80}")
print(f"PC1 alone explains: {explained_var[0]*100:.1f}%")

In [ ]:
# --- Scree Plot ---

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(1, n_components + 1)
ax.bar(x, explained_var * 100, color='steelblue', alpha=0.7,
       label='Individual', edgecolor='white')
ax2 = ax.twinx()
ax2.plot(x, cumulative_var * 100, 'o-', color='crimson', lw=2,
         markersize=6, label='Cumulative')
ax2.axhline(80, color='gray', ls='--', lw=1.5, label='80% threshold')
ax2.axvline(n_for_80, color='green', ls=':', lw=2, alpha=0.7,
            label=f'{n_for_80} components')

ax.set_xlabel('Principal Component', fontsize=12)
ax.set_ylabel('Explained Variance (%)', fontsize=12, color='steelblue')
ax2.set_ylabel('Cumulative Explained Variance (%)', fontsize=12, color='crimson')
ax.set_title('Scree Plot: PCA on Stock Return Covariance Matrix',
             fontsize=14, fontweight='bold')

# Combine legends
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=10)

ax.set_xticks(x)
plt.tight_layout()
plt.show()

In [ ]:
# --- Eigenportfolio Weights ---

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

eigenvectors = pca.components_  # shape: (n_components, N_STOCKS)

for k in range(3):
    ax = axes[k]
    weights = eigenvectors[k]
    bar_colors = ['steelblue' if w >= 0 else 'coral' for w in weights]
    ax.bar(range(N_STOCKS), weights, color=bar_colors, alpha=0.7, width=0.8)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel(f'Weight', fontsize=11)
    ax.set_title(f'Eigenportfolio {k+1} — Explains {explained_var[k]*100:.1f}% of variance',
                 fontsize=12, fontweight='bold')

axes[2].set_xlabel('Stock Index', fontsize=12)
fig.suptitle('Eigenportfolio Weights (First 3 Principal Components)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Check: are PC1 weights mostly positive? (market factor)
pc1_weights = eigenvectors[0]
print(f"PC1 weights: min={pc1_weights.min():.4f}, max={pc1_weights.max():.4f}, "
      f"mean={pc1_weights.mean():.4f}")
print(f"Fraction of positive weights in PC1: {(pc1_weights > 0).mean():.2%}")

In [ ]:
# --- Correlation of PC1 with equal-weighted market return ---

pc1_ts = returns_demeaned.values @ eigenvectors[0]  # PC1 time series
equal_weight_return = returns_df.mean(axis=1)  # equal-weighted portfolio return

corr_pc1_market = np.corrcoef(pc1_ts, equal_weight_return)[0, 1]
print(f"Correlation between PC1 and equal-weighted market return: {corr_pc1_market:.4f}")
print("This confirms that PC1 captures the common market factor.")

## 6. Mean-Variance Optimization

We now construct optimal portfolios using mean-variance optimization:
- **Efficient frontier:** the set of minimum-variance portfolios at each target return
- **Max Sharpe portfolio:** highest return per unit of risk
- **Min variance portfolio:** lowest total risk regardless of return

We use the Ledoit-Wolf shrinkage estimator for the covariance matrix
to improve stability.

In [ ]:
# --- Estimate inputs ---

# Annualized expected returns (sample mean * 252)
mu = returns_df.mean().values * 252

# Ledoit-Wolf shrinkage covariance (annualized)
lw = LedoitWolf().fit(returns_df.values)
cov_lw = lw.covariance_ * 252

print(f"Ledoit-Wolf shrinkage intensity: {lw.shrinkage_:.4f}")
print(f"Expected returns range: [{mu.min():.4f}, {mu.max():.4f}]")
print(f"Covariance matrix condition number: {np.linalg.cond(cov_lw):.1f}")

N = len(mu)

In [ ]:
# --- Portfolio optimization helper functions ---

def portfolio_variance(w, cov):
    return w @ cov @ w

def portfolio_return(w, mu):
    return w @ mu

def portfolio_volatility(w, cov):
    return np.sqrt(w @ cov @ w)

def neg_sharpe(w, mu, cov, rf=0):
    excess_ret = w @ mu - rf
    vol = np.sqrt(w @ cov @ w)
    return -excess_ret / vol if vol > 1e-10 else 0

# Common setup
x0 = np.ones(N) / N
bounds = [(0.0, 0.10)] * N  # long-only, max 10% per stock
eq_constraint = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

# --- Minimum Variance Portfolio ---
result_mv = minimize(portfolio_variance, x0, args=(cov_lw,),
                     method='SLSQP', bounds=bounds,
                     constraints=[eq_constraint],
                     options={'ftol': 1e-12, 'maxiter': 1000})
w_mv = result_mv.x
ret_mv = w_mv @ mu
vol_mv = np.sqrt(w_mv @ cov_lw @ w_mv)

# --- Maximum Sharpe Portfolio ---
result_ms = minimize(neg_sharpe, x0, args=(mu, cov_lw),
                     method='SLSQP', bounds=bounds,
                     constraints=[eq_constraint],
                     options={'ftol': 1e-12, 'maxiter': 1000})
w_ms = result_ms.x
ret_ms = w_ms @ mu
vol_ms = np.sqrt(w_ms @ cov_lw @ w_ms)

print(f"Min Variance Portfolio:  Return={ret_mv:.4f}, Vol={vol_mv:.4f}, "
      f"Sharpe={ret_mv/vol_mv:.3f}")
print(f"Max Sharpe Portfolio:    Return={ret_ms:.4f}, Vol={vol_ms:.4f}, "
      f"Sharpe={ret_ms/vol_ms:.3f}")
print(f"\nMin Var: {(w_mv > 0.001).sum()} nonzero positions")
print(f"Max Sharpe: {(w_ms > 0.001).sum()} nonzero positions")

In [ ]:
# --- Efficient Frontier ---

target_returns = np.linspace(ret_mv, mu.max() * 0.9, 50)
frontier_vols = []
frontier_rets = []

for target in target_returns:
    ret_constraint = {'type': 'eq',
                      'fun': lambda w, t=target: w @ mu - t}
    result = minimize(portfolio_variance, x0, args=(cov_lw,),
                      method='SLSQP', bounds=bounds,
                      constraints=[eq_constraint, ret_constraint],
                      options={'ftol': 1e-12, 'maxiter': 1000})
    if result.success:
        frontier_vols.append(np.sqrt(result.fun))
        frontier_rets.append(target)

frontier_vols = np.array(frontier_vols)
frontier_rets = np.array(frontier_rets)

print(f"Computed {len(frontier_vols)} efficient frontier points.")

In [ ]:
# --- Plot Efficient Frontier ---

fig, ax = plt.subplots(figsize=(12, 8))

# Individual stocks
stock_vols = np.sqrt(np.diag(cov_lw))
ax.scatter(stock_vols, mu, c='gray', alpha=0.4, s=30, label='Individual Stocks')

# Efficient frontier
ax.plot(frontier_vols, frontier_rets, 'b-', lw=2.5, label='Efficient Frontier')

# Min variance portfolio
ax.scatter(vol_mv, ret_mv, c='red', s=200, zorder=5, edgecolors='black',
           label=f'Min Variance (Sharpe={ret_mv/vol_mv:.2f})')

# Max Sharpe portfolio
ax.scatter(vol_ms, ret_ms, c='lime', s=300, marker='*', zorder=5,
           edgecolors='black', linewidths=1,
           label=f'Max Sharpe (Sharpe={ret_ms/vol_ms:.2f})')

# Equal weight portfolio
w_eq = np.ones(N) / N
ret_eq = w_eq @ mu
vol_eq = np.sqrt(w_eq @ cov_lw @ w_eq)
ax.scatter(vol_eq, ret_eq, c='orange', s=150, marker='D', zorder=5,
           edgecolors='black',
           label=f'Equal Weight (Sharpe={ret_eq/vol_eq:.2f})')

# Risk parity portfolio
inv_vols = 1.0 / stock_vols
w_rp = inv_vols / inv_vols.sum()
ret_rp = w_rp @ mu
vol_rp = np.sqrt(w_rp @ cov_lw @ w_rp)
ax.scatter(vol_rp, ret_rp, c='purple', s=150, marker='s', zorder=5,
           edgecolors='black',
           label=f'Risk Parity (Sharpe={ret_rp/vol_rp:.2f})')

# Capital Market Line (from origin to max Sharpe)
cml_vols = np.linspace(0, vol_ms * 1.5, 100)
cml_rets = (ret_ms / vol_ms) * cml_vols
ax.plot(cml_vols, cml_rets, 'g--', lw=1.5, alpha=0.6, label='Capital Market Line')

ax.set_xlabel('Annualised Volatility', fontsize=13)
ax.set_ylabel('Annualised Return', fontsize=13)
ax.set_title('Efficient Frontier and Portfolio Strategies',
             fontsize=15, fontweight='bold')
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
plt.tight_layout()
plt.show()

## 7. Strategy Comparison

We compare four portfolio strategies:
1. **Equal Weight** — $w_i = 1/N$ for all $i$
2. **Max Sharpe** — MVO portfolio maximizing the Sharpe ratio
3. **Min Variance** — MVO portfolio minimizing total variance
4. **Risk Parity** — weights inversely proportional to volatility

In [ ]:
# --- Summary table ---

strategies = {
    'Equal Weight': w_eq,
    'Max Sharpe': w_ms,
    'Min Variance': w_mv,
    'Risk Parity': w_rp,
}

summary_rows = []
for name, w in strategies.items():
    ret = w @ mu
    vol = np.sqrt(w @ cov_lw @ w)
    sharpe = ret / vol
    n_positions = (w > 0.001).sum()
    max_weight = w.max()
    hhi = (w ** 2).sum()  # Herfindahl index (concentration)
    summary_rows.append({
        'Strategy': name,
        'Ann. Return': f'{ret:.4f}',
        'Ann. Vol': f'{vol:.4f}',
        'Sharpe': f'{sharpe:.3f}',
        '# Positions': n_positions,
        'Max Weight': f'{max_weight:.3f}',
        'HHI': f'{hhi:.4f}',
    })

summary_df = pd.DataFrame(summary_rows).set_index('Strategy')
print("=" * 70)
print("PORTFOLIO STRATEGY COMPARISON")
print("=" * 70)
print(summary_df.to_string())

In [ ]:
# --- Cumulative return backtest ---

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

strategy_colors = {'Equal Weight': 'orange', 'Max Sharpe': 'green',
                   'Min Variance': 'red', 'Risk Parity': 'purple'}

# Panel 1: Cumulative returns
ax = axes[0]
for name, w in strategies.items():
    daily_port_ret = returns_df.values @ w
    cum_ret = (1 + pd.Series(daily_port_ret, index=returns_df.index)).cumprod()
    ax.plot(cum_ret.index, cum_ret.values, lw=2,
            color=strategy_colors[name], label=name)

ax.set_ylabel('Cumulative Return (Growth of $1)', fontsize=12)
ax.set_title('Cumulative Portfolio Returns', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

# Panel 2: Weight distribution
ax = axes[1]
x_pos = np.arange(N)
width = 0.2
for idx, (name, w) in enumerate(strategies.items()):
    w_sorted = np.sort(w)[::-1]
    ax.bar(x_pos + idx * width, w_sorted, width=width,
           color=strategy_colors[name], alpha=0.7, label=name)

ax.set_xlabel('Stock Rank (by weight)', fontsize=12)
ax.set_ylabel('Portfolio Weight', fontsize=12)
ax.set_title('Weight Distribution (Sorted Descending)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(-0.5, 30)  # Show top 30 for readability

plt.tight_layout()
plt.show()

In [ ]:
# --- Rolling Sharpe ratio comparison ---

rolling_window = 63  # ~3 months

fig, ax = plt.subplots(figsize=(14, 6))

for name, w in strategies.items():
    daily_ret = pd.Series(returns_df.values @ w, index=returns_df.index)
    rolling_sharpe = (daily_ret.rolling(rolling_window).mean()
                      / daily_ret.rolling(rolling_window).std()) * np.sqrt(252)
    ax.plot(rolling_sharpe.index, rolling_sharpe.values, lw=1.5,
            color=strategy_colors[name], label=name, alpha=0.8)

ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Rolling Sharpe Ratio (63-day)', fontsize=12)
ax.set_xlabel('Date', fontsize=12)
ax.set_title('Rolling Sharpe Ratio Comparison (63-day window)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 8. Key Takeaways

1. **Signal evaluation is essential before portfolio construction.** The Information Coefficient (IC)
   and its Information Ratio (ICIR) quantify how well a signal predicts forward returns. A good signal
   has IC > 0.02 and ICIR > 0.5.

2. **Combining signals improves predictive power.** IC-weighted combination gives more weight to
   better-performing signals. The composite signal typically has a higher ICIR than any individual factor.

3. **PCA extracts statistical factors from the return covariance matrix.** The first principal component
   captures the market factor (30-50% of variance). Eigenportfolios provide an orthogonal decomposition
   of the covariance structure.

4. **Mean-variance optimization provides the theoretical foundation for portfolio construction.**
   The max Sharpe portfolio ($\mathbf{w}^* \propto \Sigma^{-1}\mu$) is optimal in theory but
   sensitive to estimation error in practice.

5. **Practical MVO requires regularization.** Ledoit-Wolf shrinkage for the covariance matrix,
   position limits, and risk parity are essential tools for producing stable, tradeable portfolios.

6. **Simple strategies are hard to beat.** Equal weight and risk parity often perform surprisingly
   well out of sample because they do not require return estimates and avoid estimation error.

7. **Diversification across uncorrelated signals is the closest thing to a free lunch.**
   The full pipeline — signal generation, combination, factor risk check, portfolio optimization,
   and execution — is the architecture used by most quantitative equity funds.